<a href="https://colab.research.google.com/github/zhanybeku/energy_consumption/blob/main/energy_consumption.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mrsimple07/energy-consumption-prediction")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'energy-consumption-prediction' dataset.
Path to dataset files: /kaggle/input/energy-consumption-prediction


In [4]:
import pandas as pd

df = pd.read_csv(f"{path}/Energy_consumption.csv")
df.head()

,Timestamp,Temperature,Humidity,SquareFootage,Occupancy,HVACUsage,LightingUsage,RenewableEnergy,DayOfWeek,Holiday,EnergyConsumption
0,2022-01-01 00:00:00,25.139433,43.431581,1565.693999,5,On,Off,2.774699,Monday,No,75.364373
1,2022-01-01 01:00:00,27.731651,54.225919,1411.064918,1,On,On,21.831384,Saturday,No,83.401855
2,2022-01-01 02:00:00,28.704277,58.907658,1755.715009,2,Off,Off,6.764672,Sunday,No,78.270888
3,2022-01-01 03:00:00,20.080469,50.371637,1452.316318,1,Off,On,8.623447,Wednesday,No,56.519850
4,2022-01-01 04:00:00,23.097359,51.401421,1094.130359,9,On,Off,3.071969,Friday,No,70.811732


In [5]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

# Feature engineering
df['Hour'] = pd.to_datetime(df['Timestamp']).dt.hour
df['Month'] = pd.to_datetime(df['Timestamp']).dt.month

# Encode categoricals
df['HVACUsage'] = df['HVACUsage'].map({'On': 1, 'Off': 0})
df['LightingUsage'] = df['LightingUsage'].map({'On': 1, 'Off': 0})
df['Holiday'] = df['Holiday'].map({'Yes': 1, 'No': 0})
df = pd.get_dummies(df, columns=['DayOfWeek'])

# Chronological split
df = df.sort_values('Timestamp')
split = int(len(df) * 0.8)
train, test = df.iloc[:split], df.iloc[split:]

features = [c for c in df.columns if c not in ['Timestamp', 'EnergyConsumption']]
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(train[features], train['EnergyConsumption'])

preds = model.predict(test[features])
print("MAE:", mean_absolute_error(test['EnergyConsumption'], preds))

MAE: 4.33428467338214


In [6]:
print("Mean:", df['EnergyConsumption'].mean())
print("Min:", df['EnergyConsumption'].min())
print("Max:", df['EnergyConsumption'].max())

Mean: 77.05587286869279
Min: 53.26327834004948
Max: 99.20111959032307


In [7]:
import numpy as np
mape = np.mean(np.abs((test['EnergyConsumption'] - preds) / test['EnergyConsumption'])) * 100
print(f"MAPE: {mape:.2f}%")

MAPE: 5.60%


In [8]:
train_preds = model.predict(train[features])
print("Train MAE:", mean_absolute_error(train['EnergyConsumption'], train_preds))
print("Test MAE:", mean_absolute_error(test['EnergyConsumption'], preds))

Train MAE: 1.5883412186434984
Test MAE: 4.33428467338214
